# Interactive 3D viewer for comparison windows

Renders the event windows that `main.py` feeds to the distance metric, as
interactive 3D point clouds (drag to rotate, scroll to zoom, click legend
entries to toggle traces).

Everything is driven by **`config.yaml`**, so what you see here is the same
data the pipeline compares -- same baseline, same window scheme, same feature
scaling, same modifiers.

**Two things worth knowing before you read the plots:**

1. Points are drawn in **feature space** (post `window_features`): polarity is
   dropped, `t` is shifted so the window starts at 0, and each axis is divided
   by its `feature_scales` entry. That is what the metric actually sees -- a
   raw `x, y, t` plot would hide any scaling problem, which is usually the
   thing making results look strange.
2. The scene uses `aspectmode="data"`, so relative axis lengths are **real**.
   If the cloud looks like a flat slab, the metric is seeing a flat slab.

This notebook only ever *reads*. It deliberately does not call
`PipelineConfig.from_yaml()`, because that creates a new `output/run_*`
directory as a side effect.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate the project root whether the kernel starts in the repo root or in notebook/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"config.yaml not found starting from {Path.cwd()}")

# The other notebooks here run on the `dvs_general_py39` kernel, where this
# project may not be pip-installed. Point at ./src directly so the imports
# below work under any kernel that has numpy / h5py / pyyaml / plotly.
for path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import plotly.graph_objects as go
import yaml

from event_data_toolbox.event_data_manager import EventDataManager
from event_data_toolbox.event_windows_management import EventWindowsManager
from event_analysis_toolbox.feature_preprocessing import window_features
from event_analysis_toolbox.event_modifiers import ModifierContext, build_pipelines

# If figures do not appear, force a renderer:
#   import plotly.io as pio; pio.renderers.default = "vscode"    # or "notebook", "browser"

print("project root:", PROJECT_ROOT)

## 1. Load `config.yaml`

Read directly with `yaml.safe_load` -- no run directory is created.

In [ ]:
with open(CONFIG_PATH, "r") as config_file:
    CFG = yaml.safe_load(config_file)

REAL_PATH      = Path(CFG["real_data_path"])
V2E_PATH       = Path(CFG["v2e_data_path"])
BASELINE_START = int(CFG["baseline_start"])
BASELINE_END   = int(CFG["baseline_end"])
FEATURE_NAMES  = CFG.get("feature_names")
FEATURE_SCALES = CFG.get("feature_scales")
SENSOR         = CFG.get("sensor")
SEED           = CFG.get("seed")
WINDOWS_CFG    = CFG.get("windows") or {}
SCHEMES        = CFG.get("window_schemes") or [{"name": "consecutive", "stride": None}]

print(f"metric          : {CFG.get('metric')}")
print(f"baseline        : [{BASELINE_START}, {BASELINE_END}) us  (width {BASELINE_END - BASELINE_START})")
print(f"feature_scales  : {FEATURE_SCALES}")
print(f"sensor          : {SENSOR}")
print(f"windows         : {WINDOWS_CFG}")
print(f"schemes         : {[s.get('name') for s in SCHEMES]}")
print(f"real h5         : {REAL_PATH.name}")
print(f"v2e  h5         : {V2E_PATH.name}")

## 2. Load the streams and build one scheme's windows

Uses the pipeline's own `EventWindowsManager`, so window boundaries match a
real run exactly. Building the time index reads the full `t` column once, which
takes a few seconds on these files.

Set `SCHEME_NAME` to any scheme defined under `window_schemes` in
`config.yaml`; the names available are printed in section 1.

In [ ]:
SCHEME_NAME = "consecutive"   # <-- change to "periodic" to inspect the other scheme

data_manager = EventDataManager()
real_events = data_manager.load_event_data_h5(REAL_PATH, dataset_name="events", data_key="real_data")
v2e_events  = data_manager.load_event_data_h5(V2E_PATH,  dataset_name="events", data_key="v2e_data")
print(f"real: {real_events.shape[0]:,} events, dtype {real_events.dtype.names}")
print(f"v2e : {v2e_events.shape[0]:,} events, dtype {v2e_events.dtype.names}")

scheme = next(
    (s for s in SCHEMES if (s.get("name") or "scheme") == SCHEME_NAME),
    SCHEMES[0],
)
STRIDE = scheme.get("stride")

generated = EventWindowsManager(real_events, v2e_events).generate(
    BASELINE_START,
    BASELINE_END,
    int(WINDOWS_CFG.get("n_real_windows", 9)),
    int(WINDOWS_CFG.get("n_v2e_windows", 10)),
    STRIDE,
)
baseline_window = generated["baseline"]
real_windows    = generated["real"]
v2e_windows     = generated["v2e"]

print(f"\nscheme '{SCHEME_NAME}' settings: {generated['settings']}")
print(f"baseline    : [{baseline_window.start}, {baseline_window.end}) us, n={baseline_window.n_events:,}")
print(f"real windows: {len(real_windows)}   v2e windows: {len(v2e_windows)}")

## 3. Feature-extent report

The cheapest explanation for a strange distance curve is that one axis
dominates the others after scaling. This prints each feature's span before and
after `feature_scales` is applied.

The `share` column is the one to read: it is each axis's fraction of the total
scaled span. With `t: 42`, a window of width `W` microseconds spans `W / 42`
units in time, against up to `sensor.width` in `x` -- so the time axis is
usually the smallest contributor. Whether that weighting is what you want is a
modelling decision, but it should be a deliberate one.

In [ ]:
def extent_report(window, label: str) -> None:
    """Print per-feature raw vs scaled spans for one window."""
    raw, names = window_features(
        window.events, feature_names=FEATURE_NAMES,
        feature_scales=None, time_origin=window.start,
    )
    scaled, _ = window_features(
        window.events, feature_names=FEATURE_NAMES,
        feature_scales=FEATURE_SCALES, time_origin=window.start,
    )
    print(f"\n{label}  [{window.start}, {window.end}) us   n={len(raw):,}")
    print(f"  {'feat':>5} {'raw min':>12} {'raw max':>12} {'raw span':>12} "
          f"{'scaled span':>13} {'share':>7}")
    scaled_spans = scaled.max(axis=0) - scaled.min(axis=0)
    total = scaled_spans.sum() or 1.0
    for i, name in enumerate(names or []):
        print(f"  {name:>5} {raw[:, i].min():12.1f} {raw[:, i].max():12.1f} "
              f"{raw[:, i].max() - raw[:, i].min():12.1f} "
              f"{scaled_spans[i]:13.2f} {scaled_spans[i] / total:6.1%}")


extent_report(baseline_window, "baseline (real)")
extent_report(real_windows[0], "real[0]")
extent_report(v2e_windows[0], "v2e[0]")

## 4. Event count per window

A window that lands in a gap (few or no events) produces a NaN or a wildly
out-of-family distance. Check this before reading anything into the 3D plots.

In [ ]:
real_counts = np.array([w.n_events for w in real_windows])
v2e_counts  = np.array([w.n_events for w in v2e_windows])

count_fig = go.Figure()
count_fig.add_trace(go.Scatter(
    x=[w.start for w in real_windows], y=real_counts,
    mode="lines+markers", name="real", line=dict(color="#2ca02c"),
))
count_fig.add_trace(go.Scatter(
    x=[w.start for w in v2e_windows], y=v2e_counts,
    mode="lines+markers", name="v2e", line=dict(color="#d62728"),
))
count_fig.add_hline(
    y=baseline_window.n_events, line_dash="dash", line_color="#1f77b4",
    annotation_text="baseline", annotation_position="top left",
)
count_fig.update_layout(
    title=f"Events per window - scheme '{SCHEME_NAME}' (stride={generated['settings']['stride']} us)",
    xaxis_title="window start (us)", yaxis_title="event count",
    height=380, margin=dict(l=60, r=30, t=60, b=50), template="plotly_white",
)
count_fig.show()

for label, counts in (("real", real_counts), ("v2e", v2e_counts)):
    thin = np.flatnonzero(counts < 2)
    print(f"{label}: min={counts.min():,}  max={counts.max():,}  median={int(np.median(counts)):,}"
          f"  windows with <2 events: {thin.tolist() if thin.size else 'none'}")

## 5. 3D viewer helpers

`SCALED = True` draws feature space (what the metric sees). Flip it to `False`
to see raw sensor coordinates.

`aspectmode="data"` keeps axis proportions honest. `"cube"` stretches every
axis to fill the box -- easier to look at, but it hides exactly the kind of
axis imbalance you are hunting for.

In [ ]:
MAX_POINTS = 20_000   # per trace; plotly stays responsive to roughly this many

PALETTE = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd", "#ff7f0e", "#17becf", "#8c564b"]


def axis_columns(names) -> tuple[int, int, int]:
    """Column indices to plot as (X, Y, Z), putting time on Z when present."""
    lowered = [n.lower() for n in (names or ())]
    order = [lowered.index(t) for t in ("x", "y", "t") if t in lowered]
    return tuple(order) if len(order) == 3 else (0, 1, 2)


def window_points(events, *, start, scaled=True, absolute_time=False, max_points=MAX_POINTS, rng=None):
    """Feature-space points for one window, uniformly subsampled for rendering."""
    features, names = window_features(
        events,
        feature_names=FEATURE_NAMES,
        feature_scales=FEATURE_SCALES if scaled else None,
        time_origin=None if absolute_time else start,
    )
    n_total = len(features)
    if max_points and n_total > max_points:
        rng = rng or np.random.default_rng(0)
        features = features[np.sort(rng.choice(n_total, max_points, replace=False))]
    return features, names, n_total


def plot_windows_3d(traces, *, title, scaled=True, absolute_time=False,
                    aspectmode="data", marker_size=1.6, opacity=0.65, height=760):
    """traces: iterable of (label, events, window_start, color)."""
    fig = go.Figure()
    for label, events, start, color in traces:
        points, names, n_total = window_points(
            events, start=start, scaled=scaled, absolute_time=absolute_time,
        )
        ix, iy, iz = axis_columns(names)
        shown = f"{len(points):,}/{n_total:,}" if len(points) < n_total else f"{n_total:,}"
        fig.add_trace(go.Scatter3d(
            x=points[:, ix], y=points[:, iy], z=points[:, iz],
            mode="markers", name=f"{label}  [{shown}]",
            marker=dict(size=marker_size, color=color, opacity=opacity),
            hovertemplate="x=%{x:.1f}<br>y=%{y:.1f}<br>t=%{z:.1f}<extra>" + label + "</extra>",
        ))
    space = "scaled" if scaled else "raw"
    t_label = "t abs" if absolute_time else "t from window start"
    fig.update_layout(
        title=f"{title}<br><sub>{space} feature space | aspectmode={aspectmode}</sub>",
        height=height, margin=dict(l=0, r=0, t=80, b=0), template="plotly_white",
        legend=dict(itemsizing="constant", yanchor="top", y=0.95),
        scene=dict(
            xaxis_title=f"x ({space})", yaxis_title=f"y ({space})",
            zaxis_title=f"{t_label} ({space})", aspectmode=aspectmode,
        ),
    )
    return fig

## 6. Baseline vs one real and one v2e window

The core diagnostic. The baseline is fixed; pick which comparison windows to
overlay. Click legend entries to isolate a single cloud.

If the v2e cloud sits somewhere the real cloud does not, that is an alignment
problem, not a metric problem.

In [ ]:
REAL_INDEX = 0     # index into real_windows
V2E_INDEX  = 0     # index into v2e_windows
SCALED     = True  # False -> raw sensor coordinates

real_pick = real_windows[REAL_INDEX]
v2e_pick  = v2e_windows[V2E_INDEX]

fig_pair = plot_windows_3d(
    [
        ("baseline (real)", baseline_window.events, baseline_window.start, PALETTE[0]),
        (f"real[{REAL_INDEX}] @{real_pick.start}us", real_pick.events, real_pick.start, PALETTE[1]),
        (f"v2e[{V2E_INDEX}] @{v2e_pick.start}us",    v2e_pick.events,  v2e_pick.start,  PALETTE[2]),
    ],
    title="Baseline vs one real and one v2e window",
    scaled=SCALED,
)
fig_pair.show()

## 7. What a modifier actually does

Your config sweeps `add_noise` from 1000 to 15000 events. `add_noise` spreads
noise **uniformly over the whole sensor** (`0..width`, `0..height`) and across
the window's full time range, while real events are concentrated on moving
edges. Overlaying the two shows how much of the distance change is just the
bounding box filling up.

In [ ]:
pipelines = build_pipelines(CFG.get("modifiers"))
print("pipelines:", [p.name for p in pipelines] or "(none configured)")

PIPELINE_INDEX = -1        # -1 = last / strongest sweep value
TARGET = real_windows[0]   # window the modifier is applied to

if pipelines:
    pipeline = pipelines[PIPELINE_INDEX]
    context = ModifierContext(
        rng=np.random.default_rng(SEED),
        window_start=TARGET.start,
        window_end=TARGET.end,
        sensor=SENSOR,
    )
    modified = pipeline.apply(np.asarray(TARGET.events), context)
    print(f"'{pipeline.name}': {TARGET.n_events:,} -> {len(modified):,} events")

    fig_modifier = plot_windows_3d(
        [
            ("baseline (real)", baseline_window.events, baseline_window.start, PALETTE[0]),
            (f"real[0] original", TARGET.events, TARGET.start, PALETTE[1]),
            (f"real[0] + {pipeline.name}", modified, TARGET.start, PALETTE[3]),
        ],
        title=f"Modifier effect: {pipeline.name}",
        scaled=SCALED,
    )
    fig_modifier.show()
else:
    print("No modifiers configured; skipping.")

## 8. Window sequence over absolute time

Several consecutive windows drawn on a shared absolute time axis, so you can
see the stream evolve and spot gaps, bursts, or a real/v2e drift that a single
window pair would not reveal.

In [ ]:
N_SEQUENCE = 6    # how many windows per stream
SEQ_STEP   = 1    # take every SEQ_STEP-th window

seq_real = real_windows[: N_SEQUENCE * SEQ_STEP : SEQ_STEP]
seq_v2e  = v2e_windows[: N_SEQUENCE * SEQ_STEP : SEQ_STEP]

sequence_traces = [
    (f"real @{w.start}us", w.events, w.start, PALETTE[1]) for w in seq_real
] + [
    (f"v2e @{w.start}us", w.events, w.start, PALETTE[2]) for w in seq_v2e
]

fig_seq = plot_windows_3d(
    sequence_traces,
    title=f"Window sequence on a shared time axis - scheme '{SCHEME_NAME}'",
    scaled=SCALED,
    absolute_time=True,
)
fig_seq.show()

## 9. Export a standalone HTML copy

Writes a fully self-contained file (plotly.js inlined, ~3 MB) that opens in any
browser with no kernel and no network.

In [ ]:
# Pick which figure to export: "pair", "modifier", or "sequence".
EXPORT_FIGURE = "pair"

_figures = {"pair": "fig_pair", "modifier": "fig_modifier", "sequence": "fig_seq"}
figure = globals().get(_figures[EXPORT_FIGURE])
if figure is None:
    raise NameError(f"Run the cell that builds '{EXPORT_FIGURE}' first.")

EXPORT_DIR = PROJECT_ROOT / "output" / "viz"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_path = EXPORT_DIR / f"windows_3d_{SCHEME_NAME}_{EXPORT_FIGURE}.html"

figure.write_html(export_path, include_plotlyjs=True, full_html=True)
print("wrote", export_path, f"({export_path.stat().st_size / 1e6:.1f} MB)")

## What to look for

Working through these in order usually explains a strange distance curve:

- **Event counts (section 4)** -- any window with `<2` events yields `NaN`, but
  the bigger trap is a *density mismatch between streams*: if v2e windows carry
  many times the events of the real baseline, the distance curve tracks
  density, not geometry. Compare the two series before reading anything else.
- **Axis share (section 3)** -- if one feature holds the great majority of the
  scaled span, the metric is effectively measuring that one axis. `t: 42` gives
  time ~119 units against `x` up to 1280.
- **Real vs v2e placement (section 6)** -- offset clouds mean a timing or
  homography issue upstream, not a metric issue. Note that your v2e file
  (`..._offset19000.h5`) already has the 19000 us shift baked in, so nothing
  further is applied at runtime.
- **Modifier scale (section 7)** -- 15000 uniform noise events against a ~9400
  event window is more noise than signal; the top of the sweep may be far past
  the regime you care about.
- **Drift across windows (section 8)** -- a slow divergence points at
  accumulating clock skew rather than per-window noise.

### If you need the full stream instead

This notebook is scoped to windows (~10k events each), where plotly is
comfortable. To fly through all ~15M events at once, use Open3D -- already a
dependency -- which opens a native GPU viewer:

```python
import open3d as o3d
import numpy as np

sub = real_events[::50]                       # decimate; 15M points is a lot
xyz = np.column_stack([sub["x"], sub["y"], sub["t"] / 42.0]).astype(np.float64)
cloud = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(xyz))
o3d.visualization.draw_geometries([cloud])    # blocking native window
```

It handles millions of points smoothly, but it opens a separate blocking
window and will not embed in the notebook.